## 第 2 课：CuTe 基础与 Minimal GEMM Kernel

> 对应原文：笔记 (1)。从零用 CuTe 写一个单 MMA 指令 16×8×8 的 GEMM kernel，并完成验证、性能测试和 PTX/SASS 分析。
> 环境：CUTLASS 4.1.0，SM90。

## 学习目标

- 理解 CuTe 的两大关键组件：**Tensor 与 Layout**；
- 会用 `local_tile`、`make_tiled_mma`（partition / partition_fragment）、Copy API；
- 写出一个单指令 MMA 的 Minimal GEMM kernel + Pytorch binding；
- 会用 NCU / ncu 和 PTX / SASS 分析 kernel。

## 1. CuTe 基础组件

### 1.1 Tensor 和 Layout

CuTe 的 Tensor 和 PyTorch Tensor 类似：表示一个张量的存储对象，并提供重载方法方便计算。

- **Layout** = Shape + Stride，表达"张量坐标 ↔ 内存 offset"的映射；
- 格式：`shape : stride`，shape 和 stride 都可以是 tuple 或**嵌套的 tuple**；
- 可嵌套性是 CuTe 不同于 PyTorch Tensor 的重要特点。

![CuTe 的第 1 种 Layout —— Tensor Layout](assets/figs/fig_01_CuTe_的第_1_种_Layout____Tensor_Layout.png)

![CuTe 中的嵌套 Layout](assets/figs/fig_02_CuTe_中的嵌套_Layout.png)

和 PyTorch 的两个不同点：

1. `make_tensor(data_ptr, shape, stride)` 不提供 stride 时，默认 **left-major**；PyTorch 默认 **right-major**；
2. 嵌套 Layout 支持更复杂的 Tensor pattern。

In [ ]:
Tensor mA = make_tensor(make_gmem_ptr((T*)Aptr),
                        make_shape(m, k),
                        make_stride(k, Int<1>{}));  // (M, K)

CuTe 中张量的维度习惯称为 **mode**（最左侧是 0th mode），嵌套 Layout 的各个维度称为 **sub mode**；用 `size(tensor)` 获取各维度大小。

![make_tensor API](assets/figs/fig_03_CuTe_API____make_tensor.png)

### 1.2 Tiling API：local_tile

把 Tensor 按 tile shape 切成若干小 Tensor，用坐标取出其中一个：

In [ ]:
Tensor gA = local_tile(mA, make_shape(Int<kTileM>{}, Int<kTileK>{}), make_coord(0, 0));

高维 tiler + Step 在指定维度分块：

In [ ]:
auto tiler = make_tile(Int<kTileM>{}, Int<kTileN>{}, Int<kTileK>{});
auto coord = make_coord(0, 0, 0);
Tensor gA = local_tile(mA, tiler, coord, Step<_1, X, _1>{});

> `make_tile` / `make_coord` / `make_shape` / `make_stride` 返回的都是 `cute::tuple`；Tile、Coord、Shape、Stride、Step 都是它的别名。

![local_tile API](assets/figs/fig_04_CuTe_API____local_tile.png)

### 1.3 MMA API

MMA = Matrix Multiply-Accumulate，`D = A * B + C`。Tensor Core 提供若干固定 shape 的 MMA 指令，这里用 **16×8×8**（所有精度 FP16）：

In [ ]:
using MMA_op = SM80_16x8x8_F16F16F16F16_TN;

对应的 PTX 指令（一个 warp 的 32 个线程协同完成，每线程 4 个 A、2 个 B、4 个 C/D 元素）：

In [ ]:
mma.sync.aligned.m16n8k8.row.col.f16.f16.f16.f16
    {%Rd0, %Rd1}, {%Ra0, %Ra1}, {%Rb0}, {%Rc0, %Rc1};

手搓 PTX 需要按映射关系让每个线程取对元素、放对寄存器，极其繁琐。CuTe 的 MMA API 用 Layout 代数帮你建立了这些映射：

In [ ]:
using TiledMMA = decltype(make_tiled_mma(MMA_op{}));

TiledMMA tiled_mma;
ThrMMA thr_mma = tiled_mma.get_slice(tid);

Tensor tCgA = thr_mma.partition_A(gA);  // (MMA, MMA_M, MMA_K)
Tensor tCgB = thr_mma.partition_B(gB);  // (MMA, MMA_N, MMA_K)
Tensor tCgC = thr_mma.partition_C(gC);  // (MMA, MMA_M, MMA_N)

Tensor tCrA = thr_mma.partition_fragment_A(gA);  // 寄存器版，shape 同 partition_A
Tensor tCrB = thr_mma.partition_fragment_B(gB);
Tensor tCrC = thr_mma.partition_fragment_C(gC);

![Minimal GEMM kernel 中的 Tiled MMA（A/B/C 每个元素 TxVy = 线程 x 的第 y 个数据）](assets/figs/fig_04_MMA_指令的_MN_Layout_示意图.png)

命名规范：`tCgA` 中 **t**=tiling，**C**=从计算 C 的 MMA tile 出来，**g/r**=数据在 global memory / register，**A**=矩阵名。`txgy`/`txry` 是常见变体。

### 1.4 Copy API

数据从 GMEM 拷到寄存器。`AutoVectorizingCopy` 自动选取最大的连续数据长度（最多 128 bits/指令）：

In [ ]:
auto copy_atom = AutoVectorizingCopy{};
copy(copy_atom, tCgA, tCrA);

![GMEM 到 Register 的拷贝](assets/figs/fig_06_GMEM_到_Register_的拷贝.png)

## 2. 编写 Minimal GEMM kernel

算子规格：单指令 16×8×8，1 个 block、32 个线程，tile shape 等于 MMA atom shape，SMEM 为 0。

### 2.1 Kernel Spec 参数类

In [ ]:
template <typename T_, int kTileM_ = 16, int kTileN_ = 8, int kTileK_ = 8>
struct KernelSpec {
    using T = T_;
    static constexpr int kTileM = kTileM_;
    static constexpr int kTileN = kTileN_;
    static constexpr int kTileK = kTileK_;

    using MMA_op = SM80_16x8x8_F16F16F16F16_TN;
    using TiledMMA = decltype(make_tiled_mma(MMA_op{}));

    static constexpr int kThreadNum = size(TiledMMA{});
    static constexpr int kShmSize = 0;
};

### 2.2 kernel 代码（核心流程）

In [ ]:
// 1) 建立三个矩阵的 Tensor 表示（行连续，stride 最后一维为 1）
Tensor mA = make_tensor(make_gmem_ptr((T*)Aptr), make_shape(m, k), make_stride(k, Int<1>{}));
Tensor mB = make_tensor(make_gmem_ptr((T*)Bptr), make_shape(n, k), make_stride(k, Int<1>{}));
Tensor mC = make_tensor(make_gmem_ptr((T*)Cptr), make_shape(m, n), make_stride(n, Int<1>{}));

// 2) 取出本 block 的分块（单指令场景只有一个 tile，coord = (0,0,0)）
auto tiler = make_tile(Int<kTileM>{}, Int<kTileN>{}, Int<kTileK>{});
Tensor gA = local_tile(mA, tiler, make_coord(0, 0, 0), Step<_1, X, _1>{});
Tensor gB = local_tile(mB, tiler, make_coord(0, 0, 0), Step< X, _1, _1>{});
Tensor gC = local_tile(mC, tiler, make_coord(0, 0, 0), Step<_1, _1, X>{});

// 3) MMA 分块
TiledMMA tiled_mma;
ThrMMA thr_mma = tiled_mma.get_slice(tid);
Tensor tCgA = thr_mma.partition_A(gA);
Tensor tCgB = thr_mma.partition_B(gB);
Tensor tCgC = thr_mma.partition_C(gC);
Tensor tCrA = thr_mma.partition_fragment_A(gA);
Tensor tCrB = thr_mma.partition_fragment_B(gB);
Tensor tCrC = thr_mma.partition_fragment_C(gC);

// 4) 拷贝 + 计算（IsGemm 时清空累加器，MMA 时拷贝 C）
auto copy_atom = AutoVectorizingCopy{};
copy(copy_atom, tCgA, tCrA);
copy(copy_atom, tCgB, tCrB);
if constexpr (IsGemm) clear(tCrC);
else copy(copy_atom, tCgC, tCrC);

gemm(tiled_mma, tCrC, tCrA, tCrB, tCrC);

// 5) 结果写回 GMEM
copy(copy_atom, tCrC, tCgC);

## 3. 使用 Minimal GEMM kernel

### 3.1 Pytorch binding（libtorch + pybind11）

In [ ]:
torch::Tensor c;
bool is_gemm;
if (!_c.has_value()) {
    auto options = torch::TensorOptions().dtype(torch_acc_type).device(torch::kCUDA);
    c = torch::empty({M, N}, options);
    is_gemm = true;
} else {
    c = _c.value();
    is_gemm = false;
}

BOOL_SWITCH(is_gemm, IsGemm, [&] {
    cudaEventRecord(start, stream);
    minimal_gemm<Spec, IsGemm><<<grid, block, shm_size, stream>>>(
        reinterpret_cast<AccType*>(c.data_ptr()),
        reinterpret_cast<ComputeType*>(a.data_ptr()),
        reinterpret_cast<ComputeType*>(b.data_ptr()),
        M, N, K);
    cudaEventRecord(stop, stream);
});

Python 侧用 PyTorch 即时编译接口加载动态库：

In [ ]:
a = torch.randn(M, K, device="cuda", dtype=torch.half)
b = torch.randn(N, K, device="cuda", dtype=torch.half)
c = torch.randn(M, N, device="cuda", dtype=torch.half)

kernel_output = lib.minimal_gemm(a, b, None)   # Case 1: MM
kernel_output = lib.minimal_gemm(a, b, c)      # Case 2: MMA

### 3.2 精度验证与性能测试

精度：以 PyTorch 结果为基准，比较 Max Diff / Mean Diff / 相对误差（RE < 0.001 即通过）。性能：用 CUDA Event 计时（只测 kernel 执行时间，不含 launch）。

------------------------------------------ M=16, N=8, K=8 ------------------------------------------
Block Size: (32, 1, 1) | Grid Size: (1, 1, 1) | Shared Memory Size: 0 Bytes
Kernel execution time: 0.008 ms
--------------- Result: Success, Max diff = 0.00000, Mean diff = 0.00000, RE = 0.00% ---------------

### 3.3 NCU / ncu 与 PTX / SASS

In [ ]:
ncu -o ncu_prof_1 --import-source 1 --set full --kernel-name "minimal_gemm" -f python minimal_gemm.py

重要观测指标：Duration、Compute/Memory Throughput、#Registers、Grid/Block size。**注意：ncu 时间（~3us）比 CUDA Event（~8us）更准**——Event 计时受 CPU launch 与 Event 之间的间隙影响。

![Nsight Compute 概览界面](assets/figs/fig_08_Nsight_Compute_概览界面.png)

![Event 的计时原理](assets/figs/fig_09_1_Event_的计时原理.png)

如果计算利用率高而访存利用率低 → compute bound；反之 → memory bound。

Minimal GEMM 的核心 PTX 只有 6 条（2 次 load、1 次 mma、2 次 store，前面是地址计算）：

In [ ]:
ld.global.u32  %r5, [%rd9];
ld.global.u32  %r6, [%rd11];
ld.global.u32  %r7, [%rd15];

mma.sync.aligned.m16n8k8.row.col.f16.f16.f16.f16 {%r1, %r2},{%r3, %r4},{%r5},{%r6, %r7};

st.global.u32  [%rd11], %r1;
st.global.u32  [%rd15], %r2;

SASS（SM90）核心也只有 6 条：3 条 `LDG.E`、1 条 `HMMA.1688.F16`、2 条 `STG.E`。

![Minimal GEMM 的 SASS 代码（SM90）](assets/figs/fig_14_Minimal_GEMM_的_SASS_代码_SM90_架构_.png)

## 同时回答

1. CuTe Tensor 与 PyTorch Tensor 的两个不同点是什么？`shape : stride` 表示什么含义？
2. 16×8×8 的 FP16 MMA 指令为什么要 32 个线程协同？每个线程分别持有 A、B、C/D 各几个元素、几个寄存器？（提示：A 2 个寄存器=4 个 fp16，B 1 个寄存器=2 个 fp16）
3. 原文留了一个问题：`tCgA` 和 `tCrA` 的 shape 相同，stride 有什么不同？为什么？（提示：一个是全局内存步长 128，一个是寄存器紧凑步长 2/4）

把代码和三个答案发给我，我继续审查。